In [1]:
import os
import sys
import glob
import pickle
import shutil
import zipfile
from tqdm import tqdm

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as opt
from torch.utils.data import Dataset, DataLoader

### FILES ###
import shutil # delate directory
import zipfile # extract a directory

from sklearn.model_selection import train_test_split # train and test split

In [2]:
# DELETE A DIRECTORY FROM CONTENT

folder = "/content/Dataset"

if os.path.exists(folder):
    shutil.rmtree(folder)
    print("Delated directory")
else:
    print("No delated directory")

No delated directory


In [3]:
# EXTRACT ZIP DIRECTORIES

from google.colab import drive
drive.mount('/content/drive')

LOCAL_ROOT = "/content/Dataset"

def unzip_file(zip_path):

    os.makedirs(LOCAL_ROOT, exist_ok=True)

    zip_name = os.path.basename(zip_path)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(LOCAL_ROOT)

    print(f"{zip_name} has extracted.")

#unzip_file("/content/drive/MyDrive/DATASET_SHARP/Python_code_old.zip")
unzip_file("/content/drive/MyDrive/DATASET_SHARP/doppler_traces.zip")
#unzip_file("/content/drive/MyDrive/DATASET_SHARP/doppler_traces_S4_S5.zip")

Mounted at /content/drive
doppler_traces.zip has extracted.


In [4]:
# RETURN THE ACTIONS FOR EACH DIRECTORY

def getActions(folder_path):

    folder_name = os.path.basename(os.path.normpath(folder_path))

    actions = []

    for filename in os.listdir(folder_path):

        prefix = f"{folder_name}_"
        marker = "_stream_"

        if not filename.startswith(prefix):
            continue

        if marker not in filename:
            continue

        remaining = filename[len(prefix):]

        action = remaining.split(marker)[0]

        if action not in actions:
            actions.append(action)

    actions.sort()

    print("Actions:", actions)

    return actions

In [7]:
# LISTA DELLE CARTELLE DEL DATASET

ROOT_PATH = "/content/Dataset/doppler_traces"

folders = []

for folder_name in sorted(os.listdir(ROOT_PATH)):

    folder_path = os.path.join(ROOT_PATH, folder_name)

    # Considera solo le cartelle
    if not os.path.isdir(folder_path):
        continue

    folders.append(folder_path)

print(folders)


['/content/Dataset/doppler_traces/S1a', '/content/Dataset/doppler_traces/S1b', '/content/Dataset/doppler_traces/S1c', '/content/Dataset/doppler_traces/S2a', '/content/Dataset/doppler_traces/S2b', '/content/Dataset/doppler_traces/S3a', '/content/Dataset/doppler_traces/S4a', '/content/Dataset/doppler_traces/S4b', '/content/Dataset/doppler_traces/S5a', '/content/Dataset/doppler_traces/S6a', '/content/Dataset/doppler_traces/S6b', '/content/Dataset/doppler_traces/S7a']


In [8]:
# EXTRACT DATASET

complete_dataset = {}

for i in range(0, len(folders)):

  folder = folders[i]
  actions = getActions(folder)
  dataset_array = {}

  for action in actions:
    dataset_array[action] = []

  folder_name = os.path.basename(folder)

  for action in actions:

    for filename in os.listdir(folder):
      if filename.startswith(f"{folder_name}_{action}"):

        file_path = os.path.join(folder, filename)

        with open(file_path, "rb") as fp:
          doppler = pickle.load(fp)

        dataset_array[action].append(doppler)

  complete_dataset[folder_name] = dataset_array

Actions: ['C', 'E', 'H', 'J1', 'J2', 'L', 'R', 'S', 'W']
Actions: ['C', 'E', 'H', 'J1', 'J2', 'L', 'R', 'S', 'W']
Actions: ['C', 'E', 'H', 'J1', 'J2', 'L', 'R', 'S', 'W']
Actions: ['E', 'J', 'L', 'R', 'W']
Actions: ['E', 'J1', 'J2', 'L', 'R', 'W']
Actions: ['E', 'H', 'J1', 'J2', 'L', 'R', 'S', 'W']
Actions: ['C1', 'C2', 'E', 'H1', 'H2', 'J1', 'J2', 'L', 'R', 'S', 'W']
Actions: ['E', 'J1', 'J2', 'L', 'R', 'W']
Actions: ['C1', 'C2', 'E', 'H1', 'H2', 'J1', 'J2', 'L', 'LOS', 'R', 'W']
Actions: ['C', 'E', 'H', 'J1', 'J2', 'L', 'R', 'S', 'W']
Actions: ['E', 'J1', 'J_0', 'L', 'R', 'W']
Actions: ['C', 'E', 'H', 'J1', 'J2', 'J3', 'L1', 'L2', 'R1', 'S', 'W']


In [9]:
# WINDOW OF 1@(340 x 100)

def create_sliding_windows(complete_dataset, window_length=340, stride=340):

  X = []
  y = []
  folders = []
  streams = []

  for folder_name in complete_dataset:
    print("Cartella: ", folder_name)
    dataset = complete_dataset[folder_name]
    all_windows = {}

    for action in dataset:

      data = np.asarray(dataset[action])
      windows_activity = []
      # elements of each action
      num_streams, time_length, num_features = np.array(data).shape
      print(f"Action {action} -> Shape of data: {num_streams}, {time_length}, {num_features}")

      for stream in range(0, num_streams):

        window = []
        # se lo stream è più corto in partenza della grandezza della finestra
        if time_length < window_length:
          print("The dataset is less than window size")
          continue

        start = 0
        end = 340
        while end <= time_length:
          window = data[stream][start:end,:]
          #windows_stream.append(window)
          X.append(window)
          y.append(action)
          folders.append(folder_name)
          streams.append(stream)
          start += stride
          end += stride

        #if start < time_length and end > time_length:
          #last_window = np.array(data[stream][start:time_length,:])
          #print("Zero da aggiungere: ", window_length - last_window.shape[0])
          #zero_padding = np.zeros((window_length - last_window.shape[0], last_window.shape[1]), dtype=last_window.dtype)
          #last_window = np.vstack((last_window, zero_padding))
          #windows_stream.append(last_window)
          #print("Shape ultima finestra: ", last_window.shape)
          #print("NUmbers of zeros: ", len(zero_padding))
          #print("last window: ", last_window)

      del data

    del dataset

  return X, y, folders, streams

X, y, folders, streams = create_sliding_windows(complete_dataset)

#print("\n==========================")
#print("DATASET FINALE")
#print("==========================")

#for folder_name in complete_dataset:

    #print(f"\nCartella: {folder_name}")

    #for action in complete_dataset[folder_name]:

        #print(
            #f"  {action}: {complete_dataset[folder_name][action].shape}"
        #)


Cartella:  S1a
Action C -> Shape of data: 4, 18766, 100
Action E -> Shape of data: 4, 18700, 100
Action H -> Shape of data: 4, 19064, 100
Action J1 -> Shape of data: 4, 8700, 100
Action J2 -> Shape of data: 4, 8708, 100
Action L -> Shape of data: 4, 18842, 100
Action R -> Shape of data: 4, 19269, 100
Action S -> Shape of data: 4, 19009, 100
Action W -> Shape of data: 4, 19295, 100
Cartella:  S1b
Action C -> Shape of data: 4, 18803, 100
Action E -> Shape of data: 4, 18270, 100
Action H -> Shape of data: 4, 18707, 100
Action J1 -> Shape of data: 4, 8756, 100
Action J2 -> Shape of data: 4, 8667, 100
Action L -> Shape of data: 4, 19329, 100
Action R -> Shape of data: 4, 19253, 100
Action S -> Shape of data: 4, 18922, 100
Action W -> Shape of data: 4, 18927, 100
Cartella:  S1c
Action C -> Shape of data: 4, 18533, 100
Action E -> Shape of data: 4, 18929, 100
Action H -> Shape of data: 4, 19053, 100
Action J1 -> Shape of data: 4, 8279, 100
Action J2 -> Shape of data: 4, 8784, 100
Action L -> 

In [10]:
index = np.random.randint(0, len(X))
print("Index: ", index)
print(type(X[index]))
print("Element in X: ", X[index].shape)
print("Element in y: ", y[index])
print("Element in folders: ", folders[index])
print("Element in streams: ", streams[index])

Index:  6831
<class 'numpy.ndarray'>
Element in X:  (340, 100)
Element in y:  W
Element in folders:  S2a
Element in streams:  1


In [11]:
label_map = {
    "W": 0,
    "R": 1,
    "J1": 2,
    "J2": 2,
    "L": 3,
    "S": 4,
    "C": 5,
    "G": 6,
    "E": 7
}

class DopplerDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x = self.dataset[idx]

        # Convert to float32
        x = torch.from_numpy(sample["data"]).float()
        # Add channel dimension -> (1, 340, 100)
        x = x.unsqueeze(0)

        activity = sample["label"][0]

        y = torch.tensor(label_map[activity], dtype=torch.long)

        z = sample["folder"]

        w = sample["stream"]

        return x, y, z, w


In [12]:
# DATASET, TRAINING, TEST
dataset = [
    {
        "data": data,
        "label": label,
        "folder": folder,
        "stream": stream
    }
    for data, label, folder, stream
    in zip(X, y, folders, streams)
]

dataset_S1 = []
dataset_test_external = []

for sample in dataset:
  if sample["folder"].startswith("S1"):
    dataset_S1.append(sample)
  else:
    dataset_test_external.append(sample)

labels = []

for sample in dataset_S1:
    labels.append(sample["label"])

unique_labels = sorted({s["label"] for s in dataset_S1})
print(unique_labels)

doppler_dataset = DopplerDataset(dataset_S1)
train_S1_dataset, test_S1_dataset = train_test_split(
    doppler_dataset,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

['C', 'E', 'H', 'J1', 'J2', 'L', 'R', 'S', 'W']


In [15]:
# CONTENUTO CODICE

print("Dataset: ", dataset[0])
print(dataset[0]["data"].shape)
print("Dataset completo:", len(dataset))
print("Dataset S1:", len(dataset_S1))
print("Dataset test esterno:", len(dataset_test_external))
print("Train S1:", len(train_S1_dataset))
print("Test S1:", len(test_S1_dataset))

#print(train_S1_dataset[0]["data"].shape)
print(train_S1_dataset[0][0].shape)
from collections import Counter

labels_train = []

for sample in train_S1_dataset:
    labels_train.append(sample[1])

print(Counter(labels_train))



Dataset:  {'data': array([[0.06309573, 0.06309573, 0.06309573, ..., 0.06309573, 0.06309573,
        0.06309573],
       [0.06309573, 0.06309573, 0.06309573, ..., 0.06309573, 0.06309573,
        0.06309573],
       [0.06309573, 0.06309573, 0.06309573, ..., 0.06309573, 0.06309573,
        0.06309573],
       ...,
       [0.06309573, 0.06309573, 0.06309573, ..., 0.06309573, 0.06309573,
        0.06309573],
       [0.06309573, 0.06309573, 0.06309573, ..., 0.06309573, 0.06309573,
        0.06309573],
       [0.06309573, 0.06309573, 0.06309573, ..., 0.06309573, 0.06309573,
        0.06309573]]), 'label': 'C', 'folder': 'S1a', 'stream': 0}
(340, 100)
Dataset completo: 18908
Dataset S1: 5240
Dataset test esterno: 13668
Train S1: 4192
Test S1: 1048
torch.Size([1, 340, 100])
Counter({tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tenso

In [16]:
#Network Definition

class BaseNet(nn.Module):
    def __init__(self):
        super().__init__()

        print("Network Initialized")

        self.branch1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=5, kernel_size=2, stride=2),
            nn.ReLU())
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=3, kernel_size=1, stride=1, padding='same'),
            nn.ReLU(),
            nn.Conv2d(in_channels=3, out_channels=6, kernel_size=2, stride=1, padding='same'),
            nn.ReLU(),
            nn.Conv2d(in_channels=6, out_channels=9, kernel_size=4, stride=2, padding=1),
            nn.ReLU()
        )
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=15, out_channels=3, kernel_size=1, stride=1, padding='same'),
            nn.ReLU()
        )

        self.block2 = nn.Sequential(
            nn.Dropout(0.2),
            nn.LazyLinear(out_features=8)
        )

    def forward(self, x):
        b1 = self.branch1(x)
        print("Branch 1:", b1.shape)
        b2 = self.branch2(x)
        print("Branch 2:", b2.shape)
        b3 = self.branch3(x)
        print("Branch 3:", b3.shape)
        h = torch.cat([b1, b2, b3], dim=1)
        print("Concat:", h.shape)
        h = self.block1(h)
        print("Concat:", h.shape)
        h = h.flatten(1)
        print("Flatten:", h.shape)
        out = self.block2(h)
        print("Out:", out.shape)

        return out


In [17]:
batch_size = 64
num_workers = 0
pin_memory = torch.cuda.is_available()

train_dataloader = DataLoader(train_S1_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=pin_memory)
#valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False,
#                              num_workers=num_workers, pin_memory=pin_memory)
test_dataloader = DataLoader(test_S1_dataset, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=pin_memory)

In [18]:
model = BaseNet()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

optimizer = opt.Adam(model.parameters(), lr=0.0001)

loss_fn = nn.CrossEntropyLoss()

epochs = 5

for epoch in range(epochs):
    model.train()
    print(f"Epoch:{epoch+1}")
    train_iterator = tqdm(train_dataloader)
    for batch_x, batch_y, _, _ in train_iterator:
        y_pred = model(batch_x)

        loss = loss_fn(y_pred, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_iterator.set_description(f"Train loss: {loss.detach().cpu().numpy()}")
    model.eval()


Network Initialized
Epoch:1


  0%|          | 0/66 [00:00<?, ?it/s]

Branch 1: torch.Size([64, 1, 170, 50])
Branch 2: torch.Size([64, 5, 170, 50])


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1024.)
  return F.conv2d(


Branch 3: torch.Size([64, 9, 170, 50])
Concat: torch.Size([64, 15, 170, 50])
Concat: torch.Size([64, 3, 170, 50])
Flatten: torch.Size([64, 25500])
Out: torch.Size([64, 8])


...

Train loss: 4.9116741138277575e-05:  97%|█████████▋| 64/66 [01:23<00:02,  1.21s/it]

Branch 1: torch.Size([64, 1, 170, 50])
Branch 2: torch.Size([64, 5, 170, 50])
Branch 3: torch.Size([64, 9, 170, 50])
Concat: torch.Size([64, 15, 170, 50])
Concat: torch.Size([64, 3, 170, 50])
Flatten: torch.Size([64, 25500])
Out: torch.Size([64, 8])


Train loss: 4.851700214203447e-05:  98%|█████████▊| 65/66 [01:24<00:01,  1.17s/it]

Branch 1: torch.Size([32, 1, 170, 50])
Branch 2: torch.Size([32, 5, 170, 50])
Branch 3: torch.Size([32, 9, 170, 50])
Concat: torch.Size([32, 15, 170, 50])
Concat: torch.Size([32, 3, 170, 50])
Flatten: torch.Size([32, 25500])
Out: torch.Size([32, 8])


Train loss: 4.70306804345455e-05: 100%|██████████| 66/66 [01:24<00:00,  1.29s/it]
